# LightRAG Query System

This notebook implements the query functionality for the LightRAG system using Ollama for both embeddings and text generation. Make sure you have:
1. Run the ingestion notebook first
2. Have Ollama running locally
3. Have pulled the necessary models (`ollama pull nomic-embed-text` and `ollama pull llama2` or your preferred model)

In [8]:
import numpy as np
import pickle
import requests
from typing import List, Dict, Union

## Embedding Client
This class handles generating embeddings through Ollama for query matching

In [9]:
class OllamaEmbedding:
    def __init__(self, model_name: str = "nomic-embed-text", base_url: str = "http://localhost:11434"):
        self.model_name = model_name
        self.base_url = base_url
    
    def encode(self, texts: Union[str, List[str]]) -> np.ndarray:
        if isinstance(texts, str):
            texts = [texts]
        
        embeddings = []
        for text in texts:
            response = requests.post(
                f"{self.base_url}/api/embeddings",
                json={"model": self.model_name, "prompt": text}
            )
            embeddings.append(response.json()['embedding'])
        
        return np.array(embeddings)

## Query System
This class handles loading the index and finding relevant document chunks

In [10]:
class LightRAGQuery:
    def __init__(self, index_directory: str):
        self.model = OllamaEmbedding()
        
        # Load embeddings and documents
        self.embeddings = np.load(f"{index_directory}/embeddings.npy")
        with open(f"{index_directory}/documents.pkl", 'rb') as f:
            data = pickle.load(f)
            self.documents = data['documents']
            self.metadata = data['metadata']
    
    def find_relevant_chunks(self, query: str, k: int = 3) -> List[Dict]:
        # Create query embedding
        query_embedding = self.model.encode(query)
        if len(query_embedding.shape) > 1:
            query_embedding = query_embedding[0]
        
        # Find similar chunks
        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[-k:][::-1]
        
        # Return chunks with metadata
        return [
            {
                'content': self.documents[idx],
                'metadata': self.metadata[idx],
                'similarity': similarities[idx]
            }
            for idx in top_indices
        ]

## LLM Interface
This class handles text generation through Ollama

In [11]:
class OllamaLLM:
    def __init__(self, model_name: str = "llama2", base_url: str = "http://localhost:11434"):
        self.model_name = model_name
        self.base_url = base_url
    
    def generate_response(self, prompt: str, system_prompt: str = None) -> str:
        request_body = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False
        }
        if system_prompt:
            request_body["system"] = system_prompt
            
        response = requests.post(
            f"{self.base_url}/api/generate",
            json=request_body
        )
        return response.json()['response']

## Prompt Formatting
Helper function to format prompts with context

In [12]:
def format_query_prompt(query: str, relevant_chunks: List[Dict]) -> str:
    prompt = "Based on the following context, please provide a detailed answer. Only use information from the provided context.\n\nContext:\n"
    
    for i, chunk in enumerate(relevant_chunks, 1):
        prompt += f"\nChunk {i} (from {chunk['metadata']['filename']}):\n{chunk['content']}\n"
    
    prompt += f"\nQuestion: {query}\n\nAnswer:"
    return prompt

## Query Interface
Let's set up an interactive query interface

In [24]:
# Initialize the systems
rag_query = LightRAGQuery('lightrag_index')
llm = OllamaLLM(model_name="deepseek-r1:8b")  # or your preferred model

def query_documents(question: str, show_sources: bool = True):
    # Find relevant chunks
    chunks = rag_query.find_relevant_chunks(question)
    
    # Generate response
    prompt = format_query_prompt(question, chunks)
    response = llm.generate_response(
        prompt,
        system_prompt="You are a helpful assistant that answers questions based only on the provided context."
    )
    
    print("\nAnswer:", response)
    
    if show_sources:
        print("\nSources:")
        for chunk in chunks:
            print(f"- {chunk['metadata']['filename']} (similarity: {chunk['similarity']:.3f})")

## Try it out!
Now you can query your documents

In [ ]:
# Example query
query_documents("What are the main topics discussed in the documents?")

You can also run multiple queries in sequence:

In [ ]:
while True:
    question = input("\nEnter your question (or 'quit' to exit): ")
    if question.lower() == 'quit':
        break
    query_documents(question)